In [7]:
import os
import re
import datetime
import glob
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import holidays

plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Calibri', 'Segoe UI']
plt.rcParams['axes.unicode_minus'] = False


# ============ НАСТРОЙКИ ============
INPUT_FILE = None
OUTPUT_FOLDER = "./reports"

MIN_HOURS_FOR_AMPLITUDE = 20   # минимум часов в сутках, чтобы считать амплитуду
MIN_COMPLETENESS_PCT = 70.0    # минимум полноты месяца для YoY
MIN_FULL_DAYS_IN_GROUP = 8     # минимум полных дней в окне для стабильной нормы
MIN_FULL_DAYS_FOR_YOY = 3      # минимум полных дней для расчёта ССИ месяца
Z_THRESHOLD = 3.5              # порог Z-score
DELTA_THRESHOLD = 30.0         # порог отклонения в %
WINDOW_DAYS = 15               # ± дней вокруг целевого дня для нормы
# ===================================


COL_NAMES = [
    'road_section', 'timestamp',
    'vol_total', 'vol_direct', 'vol_reverse',
    'cars_total', 'cars_direct', 'cars_reverse',
    'small_trucks_total', 'small_trucks_direct', 'small_trucks_reverse',
    'med_trucks_total', 'med_trucks_direct', 'med_trucks_reverse',
    'large_trucks_total', 'large_trucks_direct', 'large_trucks_reverse',
    'road_trains_total', 'road_trains_direct', 'road_trains_reverse',
    'buses_total', 'buses_direct', 'buses_reverse',
    'motos_total', 'motos_direct', 'motos_reverse',
    'speed_direct', 'speed_reverse',
    'load_direct', 'load_reverse'
]


def parse_traffic_excel(filepath):
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    ws = wb.active

    records = []
    current_section = "Неизвестный участок"
    date_regex = re.compile(r"^\s*\d{1,2}[\./]\d{1,2}[\./]\d{4}\s+\d{1,2}:\d{2}(:\d{2})?\s*$")
    section_regex = re.compile(r"км\s*\d+(\+\d+)?", re.IGNORECASE)
    road_regex = re.compile(r"а/д\s*Р-?\d+", re.IGNORECASE)

    for row in ws.iter_rows(values_only=True):
        if not row:
            continue
        for cell in row[:6]:
            if cell is not None and isinstance(cell, str):
                cell_s = cell.strip()
                if section_regex.search(cell_s):
                    current_section = cell_s
                    break
                elif road_regex.search(cell_s) and "отчет" not in cell_s.lower():
                    current_section = cell_s
                    break

        first_val = row[0]
        if first_val is None:
            continue
        first_str = str(first_val).strip()

        dt_obj = None
        if isinstance(first_val, (datetime.datetime, datetime.date)):
            dt_obj = pd.to_datetime(first_val)
        elif date_regex.match(first_str):
            try:
                dt_obj = pd.to_datetime(first_str, dayfirst=True)
            except Exception:
                dt_obj = None

        if dt_obj is not None and pd.notnull(dt_obj):
            row_vals = list(row[:29])
            if len(row_vals) < 29:
                row_vals.extend([None] * (29 - len(row_vals)))
            records.append([current_section, dt_obj] + row_vals[1:29])

    wb.close()

    if not records:
        return pd.DataFrame(columns=COL_NAMES)

    df = pd.DataFrame(records, columns=COL_NAMES)

    for col in COL_NAMES[2:]:
        if df[col].dtype == object:
            df[col] = (df[col].astype(str)
                       .str.replace('\xa0', '', regex=False)
                       .str.replace(' ', '', regex=False)
                       .str.replace(',', '.', regex=False))
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df


def add_calendar_flags(df):
    years = sorted(df['timestamp'].dt.year.dropna().unique().astype(int).tolist())
    ru = holidays.Russia(years=years)
    df['is_holiday'] = df['timestamp'].dt.date.isin(ru)
    df['holiday_name'] = df['timestamp'].dt.date.map(lambda d: ru.get(d, None))
    df['is_weekend'] = df['timestamp'].dt.weekday >= 5
    return df


def guess_cause(row):
    t = str(row.get('Тип аномалии', '')).lower()

    if 'неполные сутки' in t or 'пропуск' in t:
        return 'Данные неполные: судить об интенсивности нельзя'
    if 'полнота данных' in t:
        return 'Данные неполные: месяц исключён из YoY'

    if row.get('is_holiday') and row.get('holiday_name'):
        return f"Праздник: {row['holiday_name']}"

    if 'затор' in t:
        return 'Возможный затор / ДТП / ремонт'
    if 'flatline' in t or 'залипание' in t:
        return 'Неисправность датчика (flatline)'
    if 'дисбаланс' in t:
        return 'Ремонт / ДТП на одном направлении'
    if 'профил' in t:
        return 'Сбой режима движения (погода / ремонт / ЧП)'

    if 'спад' in t:
        return 'Аномально низкая интенсивность (перекрытие / погода / массовый выезд)'
    if 'всплеск' in t:
        return 'Аномально высокая интенсивность (объезд / сезон / массовый заезд)'

    return 'Причина не определена, нужна ручная проверка'


def detect_daily_anomalies(df):
    """Возвращает (амплитудные аномалии, инциденты, пропуски)."""
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df = df.copy()
    df = add_calendar_flags(df)
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour

    amplitude_anomalies = []
    incidents = []
    gaps_list = []

    for section, sec_df in df.groupby('road_section'):

        # ---------- 1. Ежедневные инциденты (затор, дисбаланс, flatline) ----------
        for dt_date, day_df in sec_df.groupby('date'):
            day_df = day_df.sort_values('hour')
            hours_cnt = len(day_df)
            missing_cnt = int(day_df['vol_total'].isna().sum())

            if hours_cnt < 24 or missing_cnt > 0:
                gaps_list.append({
                    'Участок дороги': section, 'Дата': dt_date,
                    'Часов всего': hours_cnt,
                    'Пропущено часов': 24 - hours_cnt,
                    'Часов с NaN': missing_cnt,
                })
                if hours_cnt < MIN_HOURS_FOR_AMPLITUDE:
                    continue

            vols = day_df['vol_total'].dropna().values
            if len(vols) >= 5:
                max_consec, cur_consec, flat_val = 1, 1, None
                for i in range(1, len(vols)):
                    if vols[i] == vols[i - 1] and vols[i] > 10:
                        cur_consec += 1
                        if cur_consec > max_consec:
                            max_consec = cur_consec
                            flat_val = vols[i]
                    else:
                        cur_consec = 1
                if max_consec >= 5:
                    incidents.append({
                        'Участок дороги': section, 'Дата': dt_date,
                        'Уровень': 'Критический',
                        'Детектор': 'Датчик',
                        'Тип аномалии': 'Залипание датчика (Flatline)',
                        'Параметр': 'Интенсивность',
                        'Факт': flat_val, 'Норма': 'Вариативность',
                        'Отклонение (%)': 0.0, 'Индекс (Score)': 4.0,
                        'Описание цифрами': f"Значение не менялось {max_consec} ч подряд ({flat_val} авт./ч)"
                    })

            daytime_df = day_df[(day_df['hour'] >= 7) & (day_df['hour'] <= 21)]
            daytime_tot = daytime_df['vol_total'].sum(skipna=True)
            if daytime_tot and daytime_tot > 500:
                dir_sum = daytime_df['vol_direct'].sum(skipna=True)
                if pd.notnull(dir_sum):
                    dir_ratio = dir_sum / daytime_tot
                    if dir_ratio > 0.85 or dir_ratio < 0.15:
                        incidents.append({
                            'Участок дороги': section, 'Дата': dt_date,
                            'Уровень': 'Высокий',
                            'Детектор': 'Направление',
                            'Тип аномалии': 'Дисбаланс направлений',
                            'Параметр': 'Доля прямого напр.',
                            'Факт': round(dir_ratio * 100, 1),
                            'Норма': '45-55%',
                            'Отклонение (%)': round((dir_ratio - 0.5) * 100, 1),
                            'Индекс (Score)': 2.8,
                            'Описание цифрами': f"Доля прямого направления {dir_ratio*100:.1f}%"
                        })

            spd_vals = daytime_df['speed_direct'].dropna()
            load_vals = daytime_df['load_direct'].dropna()
            if len(spd_vals) > 0 and len(load_vals) > 0:
                min_spd = spd_vals.min()
                max_load = load_vals.max()
                if min_spd < 35 and max_load > 60:
                    jam_h = int(daytime_df.loc[daytime_df['speed_direct'] == min_spd, 'hour'].iloc[0])
                    incidents.append({
                        'Участок дороги': section, 'Дата': dt_date,
                        'Уровень': 'Критический',
                        'Детектор': 'Скорость/Загрузка',
                        'Тип аномалии': 'Экстремальный затор',
                        'Параметр': 'Мин. скорость (Прямое)',
                        'Факт': round(min_spd, 1),
                        'Норма': '> 70 км/ч',
                        'Отклонение (%)': round((min_spd - 80) / 80 * 100, 1),
                        'Индекс (Score)': 4.5,
                        'Описание цифрами': f"В {jam_h}:00 скорость {min_spd} км/ч при загрузке {max_load}%"
                    })

        # ---------- 2. Амплитудный анализ со скользящим окном ----------
        full_days_by_date = {}
        for dt_date, day_df in sec_df.groupby('date'):
            if len(day_df) >= MIN_HOURS_FOR_AMPLITUDE:
                full_days_by_date[dt_date] = day_df

        if not full_days_by_date:
            continue

        meta = []
        for dt_date, day_df in full_days_by_date.items():
            meta.append({
                'date': dt_date,
                'is_weekend': bool(day_df['is_weekend'].iloc[0]),
                'is_holiday': bool(day_df['is_holiday'].any()),
                'holiday_name': (day_df['holiday_name'].dropna().iloc[0]
                                 if day_df['holiday_name'].notna().any() else None),
            })
        meta_df = pd.DataFrame(meta).sort_values('date').reset_index(drop=True)

        for _, row in meta_df.iterrows():
            dt_date = row['date']
            is_wknd = row['is_weekend']
            is_hol = row['is_holiday']

            delta = datetime.timedelta(days=WINDOW_DAYS)
            lo, hi = dt_date - delta, dt_date + delta

            mask = (
                (meta_df['date'] >= lo) & (meta_df['date'] <= hi) &
                (meta_df['is_weekend'] == is_wknd) &
                (~meta_df['is_holiday']) &
                (meta_df['date'] != dt_date)
            )
            ref_dates = meta_df.loc[mask, 'date'].tolist()

            if len(ref_dates) < MIN_FULL_DAYS_IN_GROUP:
                continue

            ref_df = pd.concat([full_days_by_date[d] for d in ref_dates])

            ref_daily = ref_df.groupby('date')['vol_total'].sum()
            vol_med = ref_daily.median()
            vol_mad = (ref_daily - vol_med).abs().median()
            vol_mad = vol_mad if vol_mad > 0 else ref_daily.std()
            vol_mad = vol_mad if (vol_mad > 0 and pd.notnull(vol_mad)) else 1.0

            # Суточный профиль нормы — выравниваем по всем 24 часам
            norm_hourly = (ref_df.groupby('hour')['vol_total']
                           .median()
                           .reindex(range(24)))

            day_df = full_days_by_date[dt_date].sort_values('hour')
            d_vol = day_df['vol_total'].sum(skipna=True)

            # ---- Профиль (корреляция) ----
            if len(day_df) == 24:
                day_vec = day_df.set_index('hour')['vol_total'].reindex(range(24))
                if day_vec.notna().all() and norm_hourly.notna().sum() >= 12:
                    norm_filled = norm_hourly.fillna(norm_hourly.median())
                    if norm_filled.std() > 0 and day_vec.std() > 0:
                        corr = np.corrcoef(norm_filled.values, day_vec.values)[0, 1]
                        if corr < 0.70 and day_vec.sum() > 1000:
                            incidents.append({
                                'Участок дороги': section, 'Дата': dt_date,
                                'Уровень': 'Высокий',
                                'Детектор': 'Форма профиля',
                                'Тип аномалии': 'Искажение суточного профиля',
                                'Параметр': 'Корреляция с нормой',
                                'Факт': round(corr, 2), 'Норма': '> 0.85',
                                'Отклонение (%)': round((corr - 1.0) * 100, 1),
                                'Индекс (Score)': 2.5,
                                'Описание цифрами': f"Профиль нетипичен (corr={corr:.2f})"
                            })

            # ---- Амплитуда ----
            z_vol = 0.6745 * (d_vol - vol_med) / vol_mad if vol_mad > 0 else 0
            delta_p = round((d_vol - vol_med) / vol_med * 100, 1) if vol_med > 0 else 0

            if z_vol < -Z_THRESHOLD and delta_p <= -DELTA_THRESHOLD:
                amplitude_anomalies.append({
                    'Участок дороги': section, 'Дата': dt_date,
                    'Уровень': 'Критический',
                    'Детектор': 'Амплитуда',
                    'Тип аномалии': 'Аномальный спад интенсивности',
                    'Параметр': 'Суточный объем',
                    'Факт': int(d_vol), 'Норма': int(vol_med),
                    'Отклонение (%)': delta_p,
                    'Индекс (Score)': round(abs(z_vol), 1),
                    'is_holiday': is_hol,
                    'holiday_name': row['holiday_name'],
                    'is_weekend': is_wknd,
                    'Описание цифрами': f"Трафик {int(d_vol)} ниже нормы {int(vol_med)} на {abs(delta_p)}% (Z={z_vol:.1f})"
                })
            elif z_vol > Z_THRESHOLD and delta_p >= DELTA_THRESHOLD:
                amplitude_anomalies.append({
                    'Участок дороги': section, 'Дата': dt_date,
                    'Уровень': 'Высокий',
                    'Детектор': 'Амплитуда',
                    'Тип аномалии': 'Аномальный всплеск интенсивности',
                    'Параметр': 'Суточный объем',
                    'Факт': int(d_vol), 'Норма': int(vol_med),
                    'Отклонение (%)': delta_p,
                    'Индекс (Score)': round(abs(z_vol), 1),
                    'is_holiday': is_hol,
                    'holiday_name': row['holiday_name'],
                    'is_weekend': is_wknd,
                    'Описание цифрами': f"Трафик {int(d_vol)} выше нормы {int(vol_med)} на +{delta_p}% (Z={z_vol:.1f})"
                })

    amp_df = pd.DataFrame(amplitude_anomalies)
    if not amp_df.empty:
        amp_df['Возможная причина'] = amp_df.apply(guess_cause, axis=1)
        amp_df = amp_df.sort_values(by=['Участок дороги', 'Дата'])

    inc_df = pd.DataFrame(incidents)
    if not inc_df.empty:
        inc_df['Возможная причина'] = inc_df.apply(guess_cause, axis=1)
        inc_df = inc_df.sort_values(by=['Участок дороги', 'Дата'])

    gaps_df = pd.DataFrame(gaps_list)
    if not gaps_df.empty:
        gaps_df = gaps_df.sort_values(by=['Участок дороги', 'Дата'])

    return amp_df, inc_df, gaps_df


def detect_monthly_anomalies(all_data_df):
    if all_data_df.empty:
        return pd.DataFrame()

    df = add_calendar_flags(all_data_df.copy())
    df['year'] = df['timestamp'].dt.year
    df['month'] = df['timestamp'].dt.month
    df['date'] = df['timestamp'].dt.date
    df['year_month'] = df['timestamp'].dt.to_period('M').astype(str)

    monthly_stats = []
    for (sec, ym), m_df in df.groupby(['road_section', 'year_month']):
        y = m_df['year'].iloc[0]
        m = m_df['month'].iloc[0]

        tot_hours = len(m_df)
        days_in_m = pd.Period(ym).days_in_month
        exp_hours = days_in_m * 24
        completeness = (tot_hours / exp_hours) * 100

        full_days = m_df.groupby('date').filter(
            lambda d: len(d) >= MIN_HOURS_FOR_AMPLITUDE
        )

        if not full_days.empty:
            full_not_holiday = full_days.groupby('date').filter(
                lambda d: not d['is_holiday'].any()
            )
        else:
            full_not_holiday = full_days

        if not full_not_holiday.empty:
            daily_vols = full_not_holiday.groupby('date')['vol_total'].sum()
            n_full_days = len(daily_vols)
            madt = daily_vols.mean() if n_full_days >= MIN_FULL_DAYS_FOR_YOY else np.nan
        else:
            n_full_days = 0
            madt = np.nan

        avg_spd_dir = m_df['speed_direct'].mean(skipna=True)
        tot_vol = m_df['vol_total'].sum(skipna=True)
        tot_heavy = (m_df['large_trucks_total'].sum(skipna=True)
                     + m_df['road_trains_total'].sum(skipna=True))
        heavy_share = (tot_heavy / tot_vol * 100) if tot_vol and tot_vol > 0 else 0

        monthly_stats.append({
            'road_section': sec, 'year_month': ym, 'year': y, 'month': m,
            'completeness_pct': round(completeness, 1),
            'madt': round(madt, 1) if pd.notnull(madt) else np.nan,
            'n_full_days': n_full_days,
            'avg_speed_dir': round(avg_spd_dir, 1) if pd.notnull(avg_spd_dir) else 0,
            'heavy_share': round(heavy_share, 1),
            'madt_is_reliable': (completeness >= MIN_COMPLETENESS_PCT
                                 and pd.notnull(madt)
                                 and n_full_days >= MIN_FULL_DAYS_FOR_YOY)
        })

    m_df = pd.DataFrame(monthly_stats)
    monthly_anomalies = []

    for (sec, cal_month), group in m_df.groupby(['road_section', 'month']):
        reliable = group[group['madt_is_reliable']]
        if len(reliable) < 2:
            continue

        madt_med = reliable['madt'].median()
        spd_med = reliable['avg_speed_dir'].median()
        hvy_med = reliable['heavy_share'].median()

        for _, row in group.iterrows():
            ym = row['year_month']

            if not row['madt_is_reliable']:
                reason = []
                if row['completeness_pct'] < MIN_COMPLETENESS_PCT:
                    reason.append(f"полнота {row['completeness_pct']}%")
                if pd.isna(row['madt']):
                    reason.append(f"полных дней: {row['n_full_days']}")
                monthly_anomalies.append({
                    'Участок дороги': sec, 'Период': ym,
                    'Уровень': 'Информационный',
                    'Детектор': 'Полнота данных',
                    'Параметр': 'Отработано часов (%)',
                    'Факт': f"{row['completeness_pct']}%",
                    'Норма': f'>= {MIN_COMPLETENESS_PCT}%',
                    'Отклонение (%)': round(row['completeness_pct'] - 100, 1),
                    'Описание цифрами': f"YoY не считается: {', '.join(reason)}"
                })
                continue

            delta_madt = ((row['madt'] - madt_med) / madt_med) * 100 if madt_med > 0 else 0
            if abs(delta_madt) >= 25.0:
                lvl = 'Критический' if abs(delta_madt) >= 40.0 else 'Высокий'
                monthly_anomalies.append({
                    'Участок дороги': sec, 'Период': ym, 'Уровень': lvl,
                    'Детектор': 'Межгодовой YoY', 'Параметр': 'ССИ',
                    'Факт': int(row['madt']), 'Норма': int(madt_med),
                    'Отклонение (%)': round(delta_madt, 1),
                    'Описание цифрами': f"ССИ {int(row['madt'])} vs норма {int(madt_med)} ({delta_madt:+.1f}%)"
                })

            delta_hvy = row['heavy_share'] - hvy_med
            if abs(delta_hvy) >= 8.0:
                monthly_anomalies.append({
                    'Участок дороги': sec, 'Период': ym, 'Уровень': 'Высокий',
                    'Детектор': 'Состав потока YoY',
                    'Параметр': 'Доля тяжелых (%)',
                    'Факт': f"{row['heavy_share']}%", 'Норма': f"{hvy_med}%",
                    'Отклонение (%)': round((delta_hvy / hvy_med) * 100, 1) if hvy_med > 0 else 0,
                    'Описание цифрами': f"Доля тяжелых: {row['heavy_share']}% vs {hvy_med}%"
                })

            delta_spd = row['avg_speed_dir'] - spd_med
            if delta_spd <= -15.0:
                monthly_anomalies.append({
                    'Участок дороги': sec, 'Период': ym, 'Уровень': 'Критический',
                    'Детектор': 'Скорость YoY', 'Параметр': 'Средняя скорость',
                    'Факт': f"{row['avg_speed_dir']}", 'Норма': f"{spd_med}",
                    'Отклонение (%)': round((delta_spd / spd_med) * 100, 1) if spd_med > 0 else 0,
                    'Описание цифрами': f"Скорость {row['avg_speed_dir']} км/ч упала на {abs(delta_spd):.1f} ниже нормы"
                })

    res_df = pd.DataFrame(monthly_anomalies)
    if not res_df.empty:
        res_df = res_df.sort_values(by=['Участок дороги', 'Период'])
    return res_df


def find_series(df, min_len=3, max_gap_days=1):
    """Сворачивает подряд идущие однотипные аномалии в серии."""
    if df.empty:
        return pd.DataFrame()
    df = df.copy()
    df['Дата'] = pd.to_datetime(df['Дата'])
    result = []

    for (sec, atype), g in df.groupby(['Участок дороги', 'Тип аномалии']):
        g = g.sort_values('Дата').reset_index(drop=True)
        rows = [g.loc[0]]
        prev = g.loc[0, 'Дата']

        def flush(rows):
            if len(rows) >= min_len:
                dates = [r['Дата'] for r in rows]
                result.append({
                    'Участок дороги': sec,
                    'Тип аномалии': atype,
                    'Начало': min(dates).date(),
                    'Конец': max(dates).date(),
                    'Дней в серии': len(rows),
                    'Среднее отклонение (%)': round(np.mean([r['Отклонение (%)'] for r in rows]), 1),
                    'Макс. |Score|': round(max(abs(r['Индекс (Score)']) for r in rows), 1),
                    'Пример причины': rows[0].get('Возможная причина', '')
                })

        for i in range(1, len(g)):
            cur = g.loc[i, 'Дата']
            if (cur - prev).days <= max_gap_days:
                rows.append(g.loc[i])
                prev = cur
            else:
                flush(rows)
                rows = [g.loc[i]]
                prev = cur
        flush(rows)

    res = pd.DataFrame(result)
    if not res.empty:
        res = res.sort_values(by=['Участок дороги', 'Начало'])
    return res


def generate_anomaly_plots(df, daily_anomalies_df, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    if daily_anomalies_df.empty or df.empty:
        return

    df = df.copy()
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['is_weekend'] = df['timestamp'].dt.weekday >= 5
    df['month_key'] = df['timestamp'].dt.to_period('M').astype(str)

    grouped = daily_anomalies_df.groupby(['Участок дороги', 'Дата'])
    print(f"Генерация графиков (дней: {len(grouped)})...")

    hours = np.arange(24)
    for (sec, dt_d), rows in grouped:
        day_df = df[(df['road_section'] == sec) & (df['date'] == dt_d)].sort_values('hour')
        if len(day_df) == 0:
            continue

        m_key = day_df['month_key'].iloc[0]
        is_wknd = day_df['is_weekend'].iloc[0]
        anom_types = "; ".join(rows['Тип аномалии'].unique())

        ref_df = df[(df['road_section'] == sec) & (df['month_key'] == m_key) & (df['is_weekend'] == is_wknd)]
        norm_vol = ref_df.groupby('hour')['vol_total'].median().reindex(hours).fillna(0).values
        norm_spd = ref_df.groupby('hour')['speed_direct'].median().reindex(hours).fillna(0).values

        fact_vol = day_df.set_index('hour')['vol_total'].reindex(hours).values
        fact_spd = day_df.set_index('hour')['speed_direct'].reindex(hours).values

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                                        gridspec_kw={'height_ratios': [2, 1]})

        ax1.bar(hours, norm_vol, color='#BDC3C7', alpha=0.6, width=0.6,
                label='Норма (медиана)', edgecolor='#95A5A6')
        ax1.plot(hours, fact_vol, color='#E74C3C', linewidth=2.5,
                 marker='o', markersize=5, label='Факт')
        ax1.set_ylabel('Интенсивность, авт./ч', fontsize=10, fontweight='bold')
        ax1.set_title(f"{sec} | {dt_d}\nСобытия: {anom_types}",
                      fontsize=11, fontweight='bold', pad=10)
        ax1.grid(True, linestyle='--', alpha=0.5)
        ax1.legend(loc='upper left', frameon=True)

        ax2.plot(hours, norm_spd, color='#7F8C8D', linestyle='--',
                 linewidth=1.8, label='Норма скорости')
        ax2.plot(hours, fact_spd, color='#2980B9', linewidth=2.0,
                 marker='s', markersize=4, label='Факт скорости')
        ax2.axhline(40, color='#C0392B', linestyle=':', alpha=0.7,
                    label='Порог затора')
        ax2.set_ylabel('Скорость, км/ч', fontsize=10, fontweight='bold')
        ax2.set_xlabel('Час суток', fontsize=10, fontweight='bold')
        ax2.set_xticks(hours)
        ax2.set_xticklabels([f"{h:02d}" for h in hours])
        ax2.grid(True, linestyle='--', alpha=0.5)
        ax2.legend(loc='lower left', frameon=True)

        plt.tight_layout()
        safe_sec = re.sub(r'[^a-zA-Z0-9а-яА-Я+]', '_', sec)[:20]
        fname = f"{dt_d}_{safe_sec}.png"
        fig.savefig(os.path.join(output_folder, fname), dpi=110)
        plt.close(fig)


def export_anomalies_to_excel(amp_df, inc_df, series_df, gaps_df, monthly_df, output_path):
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        def write(sheet_name, df_, empty_msg):
            if not df_.empty:
                cols = [c for c in df_.columns
                        if c not in ('is_holiday', 'holiday_name', 'is_weekend')]
                df_[cols].to_excel(writer, sheet_name=sheet_name, index=False)
            else:
                pd.DataFrame([{"Инфо": empty_msg}]).to_excel(
                    writer, sheet_name=sheet_name, index=False)

        write("Аномалии суток", amp_df, "Амплитудных аномалий не обнаружено")
        write("Заторы и инциденты", inc_df, "Инцидентов не обнаружено")
        write("Серии аномалий", series_df, "Серий (3+ дней подряд) не обнаружено")
        write("Пропуски данных", gaps_df, "Пропусков нет")
        write("Аномалии месяцев YoY", monthly_df, "Аномалий месяцев не обнаружено")

    wb = openpyxl.load_workbook(output_path)
    header_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    crit_fill = PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid")
    high_fill = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")
    info_fill = PatternFill(start_color="E2EFDA", end_color="E2EFDA", fill_type="solid")
    border_thin = Border(left=Side(style='thin', color='D9D9D9'),
                         right=Side(style='thin', color='D9D9D9'),
                         top=Side(style='thin', color='D9D9D9'),
                         bottom=Side(style='thin', color='D9D9D9'))

    for ws in wb.worksheets:
        for col_idx in range(1, ws.max_column + 1):
            c = ws.cell(row=1, column=col_idx)
            c.fill = header_fill
            c.font = header_font
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        for row_idx in range(2, ws.max_row + 1):
            level_val = None
            for col_idx in range(1, ws.max_column + 1):
                c = ws.cell(row=row_idx, column=col_idx)
                c.border = border_thin
                c.alignment = Alignment(vertical="center")
                if ws.cell(row=1, column=col_idx).value == 'Уровень':
                    level_val = c.value
            fill = None
            if level_val == 'Критический':
                fill = crit_fill
            elif level_val == 'Высокий':
                fill = high_fill
            elif level_val == 'Информационный':
                fill = info_fill
            if fill:
                for col_idx in range(1, ws.max_column + 1):
                    ws.cell(row=row_idx, column=col_idx).fill = fill
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            col_letter = openpyxl.utils.get_column_letter(col[0].column)
            ws.column_dimensions[col_letter].width = min(max(max_len + 3, 12), 48)
        ws.row_dimensions[1].height = 28
    wb.save(output_path)


# ============ ЗАПУСК ============
if INPUT_FILE is None:
    candidates = sorted(glob.glob("*.xlsx") + glob.glob("*.xls"))
    if not candidates:
        raise FileNotFoundError("В текущей папке нет .xlsx файлов")
    INPUT_FILE = candidates[0]
    print(f"Использую файл: {INPUT_FILE}")

if not os.access(INPUT_FILE, os.R_OK):
    raise PermissionError(
        f"Нет доступа к файлу '{INPUT_FILE}'. Скорее всего он открыт в Excel."
    )

print("Чтение...")
full_df = parse_traffic_excel(INPUT_FILE)
print(f"Строк: {len(full_df)}")
print("Участки:", full_df['road_section'].unique().tolist())
print("Период:", full_df['timestamp'].min(), "—", full_df['timestamp'].max())

print("\n1. Аномалии суток + инциденты + пропуски...")
amp_df, inc_df, gaps_df = detect_daily_anomalies(full_df)
print(f"   Амплитудных аномалий: {len(amp_df)}")
print(f"   Инцидентов (заторы, дисбаланс, flatline): {len(inc_df)}")
print(f"   Пропусков (неполных дней): {len(gaps_df)}")

print("\n2. Серии однотипных аномалий (>=3 дней)...")
series_df = find_series(amp_df, min_len=3, max_gap_days=1)
print(f"   Серий: {len(series_df)}")

print("\n3. Аномалии месяцев (YoY)...")
monthly = detect_monthly_anomalies(full_df)
print(f"   Найдено: {len(monthly)}")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
report = os.path.join(OUTPUT_FOLDER, "traffic_anomalies_report.xlsx")
export_anomalies_to_excel(amp_df, inc_df, series_df, gaps_df, monthly, report)
print(f"\n[OK] Отчёт: {report}")

generate_anomaly_plots(full_df, amp_df, os.path.join(OUTPUT_FOLDER, "plots"))
print(f"[OK] Графики: {os.path.join(OUTPUT_FOLDER, 'plots')}")
print("\n=== ГОТОВО ===")

print("\nАмплитудные аномалии (первые 20):")
display(amp_df.head(20) if not amp_df.empty else "—")
print("\nИнциденты (первые 20):")
display(inc_df.head(20) if not inc_df.empty else "—")
print("\nСерии (первые 20):")
display(series_df.head(20) if not series_df.empty else "—")
print("\nYoY (первые 20):")
display(monthly.head(20) if not monthly.empty else "—")

Использую файл: Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2023_01.10.2026_1792159741.xlsx
Чтение...
Строк: 124380
Участки: ['км 310+000 а/д Р-298 Курск-Борисоглебск', 'км 346+000 а/д Р-298 Курск-Борисоглебск', 'км 389+000 а/д Р-298 Курск-Борисоглебск', 'км 424+450 а/д Р-298 Курск-Борисоглебск']
Период: 2023-01-01 00:59:59 — 2026-10-01 23:59:59

1. Аномалии суток + инциденты + пропуски...
   Амплитудных аномалий: 70
   Инцидентов (заторы, дисбаланс, flatline): 13
   Пропусков (неполных дней): 663

2. Серии однотипных аномалий (>=3 дней)...
   Серий: 1

3. Аномалии месяцев (YoY)...
   Найдено: 31

[OK] Отчёт: ./reports\traffic_anomalies_report.xlsx
Генерация графиков (дней: 70)...
[OK] Графики: ./reports\plots

=== ГОТОВО ===

Амплитудные аномалии (первые 20):


,Участок дороги,Дата,Уровень,Детектор,Тип аномалии,Параметр,Факт,Норма,Отклонение (%),Индекс (Score),is_holiday,holiday_name,is_weekend,Описание цифрами,Возможная причина
0,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-01-09,Критический,Амплитуда,Аномальный спад интенсивности,Суточный объем,2764,4585,-39.7,3.7,False,None,False,Трафик 2764 ниже нормы 4585 на 39.7% (Z=-3.7),Аномально низкая интенсивность (перекрытие / п...
1,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-01-12,Критический,Амплитуда,Аномальный спад интенсивности,Суточный объем,2157,4585,-53.0,4.9,False,None,False,Трафик 2157 ниже нормы 4585 на 53.0% (Z=-4.9),Аномально низкая интенсивность (перекрытие / п...
2,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-09-22,Высокий,Амплитуда,Аномальный всплеск интенсивности,Суточный объем,8742,6673,31.0,6.8,False,None,False,Трафик 8742 выше нормы 6673 на +31.0% (Z=6.8),Аномально высокая интенсивность (объезд / сезо...
3,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-11-03,Высокий,Амплитуда,Аномальный всплеск интенсивности,Суточный объем,8712,5932,46.9,3.7,False,None,False,Трафик 8712 выше нормы 5932 на +46.9% (Z=3.7),Аномально высокая интенсивность (объезд / сезо...
4,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-12-07,Высокий,Амплитуда,Аномальный всплеск интенсивности,Суточный объем,4728,3187,48.4,4.2,False,None,False,Трафик 4728 выше нормы 3187 на +48.4% (Z=4.2),Аномально высокая интенсивность (объезд / сезо...
5,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-12-08,Высокий,Амплитуда,Аномальный всплеск интенсивности,Суточный объем,5414,3166,71.0,6.0,False,None,False,Трафик 5414 выше нормы 3166 на +71.0% (Z=6.0),Аномально высокая интенсивность (объезд / сезо...
6,км 310+000 а/д Р-298 Курск-Борисоглебск,2024-01-03,Высокий,Амплитуда,Аномальный всплеск интенсивности,Суточный объем,4726,2815,67.9,4.1,True,Новогодние каникулы,False,Трафик 4726 выше нормы 2815 на +67.9% (Z=4.1),Праздник: Новогодние каникулы
7,км 310+000 а/д Р-298 Курск-Борисоглебск,2024-01-04,Высокий,Амплитуда,Аномальный всплеск интенсивности,Суточный объем,4165,2815,48.0,4.6,True,Новогодние каникулы,False,Трафик 4165 выше нормы 2815 на +48.0% (Z=4.6),Праздник: Новогодние каникулы
8,км 310+000 а/д Р-298 Курск-Борисоглебск,2024-01-05,Высокий,Амплитуда,Аномальный всплеск интенсивности,Суточный объем,5741,2802,104.9,10.9,True,Новогодние каникулы,False,Трафик 5741 выше нормы 2802 на +104.9% (Z=10.9),Праздник: Новогодние каникулы
9,км 310+000 а/д Р-298 Курск-Борисоглебск,2024-01-06,Высокий,Амплитуда,Аномальный всплеск интенсивности,Суточный объем,4218,2814,49.9,3.7,True,Новогодние каникулы,True,Трафик 4218 выше нормы 2814 на +49.9% (Z=3.7),Праздник: Новогодние каникулы



Инциденты (первые 20):


,Участок дороги,Дата,Уровень,Детектор,Тип аномалии,Параметр,Факт,Норма,Отклонение (%),Индекс (Score),Описание цифрами,Возможная причина
2,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-08-24,Высокий,Форма профиля,Искажение суточного профиля,Корреляция с нормой,0.58,> 0.85,-41.9,2.5,Профиль нетипичен (corr=0.58),Сбой режима движения (погода / ремонт / ЧП)
3,км 310+000 а/д Р-298 Курск-Борисоглебск,2025-05-09,Высокий,Форма профиля,Искажение суточного профиля,Корреляция с нормой,0.65,> 0.85,-34.5,2.5,Профиль нетипичен (corr=0.65),Сбой режима движения (погода / ремонт / ЧП)
4,км 310+000 а/д Р-298 Курск-Борисоглебск,2025-06-21,Высокий,Форма профиля,Искажение суточного профиля,Корреляция с нормой,0.69,> 0.85,-31.3,2.5,Профиль нетипичен (corr=0.69),Сбой режима движения (погода / ремонт / ЧП)
0,км 310+000 а/д Р-298 Курск-Борисоглебск,2026-03-25,Высокий,Направление,Дисбаланс направлений,Доля прямого напр.,6.50,45-55%,-43.5,2.8,Доля прямого направления 6.5%,Ремонт / ДТП на одном направлении
5,км 310+000 а/д Р-298 Курск-Борисоглебск,2026-07-04,Высокий,Форма профиля,Искажение суточного профиля,Корреляция с нормой,0.60,> 0.85,-40.4,2.5,Профиль нетипичен (corr=0.60),Сбой режима движения (погода / ремонт / ЧП)
6,км 310+000 а/д Р-298 Курск-Борисоглебск,2026-07-13,Высокий,Форма профиля,Искажение суточного профиля,Корреляция с нормой,0.67,> 0.85,-33.3,2.5,Профиль нетипичен (corr=0.67),Сбой режима движения (погода / ремонт / ЧП)
7,км 310+000 а/д Р-298 Курск-Борисоглебск,2026-07-14,Высокий,Форма профиля,Искажение суточного профиля,Корреляция с нормой,0.67,> 0.85,-33.0,2.5,Профиль нетипичен (corr=0.67),Сбой режима движения (погода / ремонт / ЧП)
8,км 310+000 а/д Р-298 Курск-Борисоглебск,2026-07-31,Высокий,Форма профиля,Искажение суточного профиля,Корреляция с нормой,-0.14,> 0.85,-114.4,2.5,Профиль нетипичен (corr=-0.14),Сбой режима движения (погода / ремонт / ЧП)
1,км 310+000 а/д Р-298 Курск-Борисоглебск,2026-08-18,Критический,Скорость/Загрузка,Экстремальный затор,Мин. скорость (Прямое),9.00,> 70 км/ч,-88.8,4.5,В 10:00 скорость 9.0 км/ч при загрузке 100.0%,Возможный затор / ДТП / ремонт
9,км 346+000 а/д Р-298 Курск-Борисоглебск,2026-01-14,Критический,Скорость/Загрузка,Экстремальный затор,Мин. скорость (Прямое),25.00,> 70 км/ч,-68.8,4.5,В 8:00 скорость 25.0 км/ч при загрузке 100.0%,Возможный затор / ДТП / ремонт



Серии (первые 20):


,Участок дороги,Тип аномалии,Начало,Конец,Дней в серии,Среднее отклонение (%),Макс. |Score|,Пример причины
0,км 310+000 а/д Р-298 Курск-Борисоглебск,Аномальный всплеск интенсивности,2024-01-03,2024-01-08,6,60.6,10.9,Праздник: Новогодние каникулы



YoY (первые 20):


,Участок дороги,Период,Уровень,Детектор,Параметр,Факт,Норма,Отклонение (%),Описание цифрами
2,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-02,Информационный,Полнота данных,Отработано часов (%),68.2%,>= 70.0%,-31.8,"YoY не считается: полнота 68.2%, полных дней: 0"
3,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-03,Информационный,Полнота данных,Отработано часов (%),67.2%,>= 70.0%,-32.8,"YoY не считается: полнота 67.2%, полных дней: 0"
4,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-04,Информационный,Полнота данных,Отработано часов (%),58.1%,>= 70.0%,-41.9,"YoY не считается: полнота 58.1%, полных дней: 0"
5,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-05,Информационный,Полнота данных,Отработано часов (%),52.7%,>= 70.0%,-47.3,"YoY не считается: полнота 52.7%, полных дней: 0"
6,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-06,Информационный,Полнота данных,Отработано часов (%),49.3%,>= 70.0%,-50.7,"YoY не считается: полнота 49.3%, полных дней: 1"
8,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-07,Критический,Межгодовой YoY,ССИ,1283,7370,-82.6,ССИ 1283 vs норма 7370 (-82.6%)
11,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-08,Критический,Межгодовой YoY,ССИ,2993,7703,-61.1,ССИ 2993 vs норма 7703 (-61.1%)
13,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-11,Высокий,Межгодовой YoY,ССИ,4281,5910,-27.6,ССИ 4281 vs норма 5910 (-27.6%)
14,км 310+000 а/д Р-298 Курск-Борисоглебск,2023-12,Высокий,Межгодовой YoY,ССИ,3320,4925,-32.6,ССИ 3320 vs норма 4925 (-32.6%)
0,км 310+000 а/д Р-298 Курск-Борисоглебск,2024-01,Высокий,Межгодовой YoY,ССИ,2824,4198,-32.7,ССИ 2824 vs норма 4198 (-32.7%)
